In [1]:
# !pip install matplotlib seaborn polars

In [2]:
# !pip install rectools[lightfm] 

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import rectools
from rectools import Columns

In [4]:
# Считываем датасеты

df_items = pd.read_csv('./dataset/KION_DATASET/data_en/items_en.csv')
df_users = pd.read_csv('./dataset/KION_DATASET/data_en/users_en.csv')
df_interactions = pd.read_csv('./dataset/KION_DATASET/interactions.csv')

In [5]:
df_users = df_users.drop("Unnamed: 0", axis=1)
df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,M,1
1,962099,age_18_24,income_20_40,M,0
2,1047345,age_45_54,income_40_60,F,0
3,721985,age_45_54,income_20_40,F,0
4,704055,age_35_44,income_60_90,F,0


In [6]:
df_items = df_items.drop("Unnamed: 0", axis=1)
df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,description,keywords,actors_translated,actors_transliterated,directors_translated,transliterated
0,10711,film,Talk to her,Hable con ella,2002.0,"drama, foreign, detective, melodrama",Spain,NaN,16.0,NaN,"Marco, a journalist, interviews the famous Tor...","Talk, her, 2002, Spain, friends, love, strong,...","Adolfo Fernández, Ana Fernández, Dario Grandin...","Adol'fo Fernandes, Ana Fernandes, Dario Grandi...",Pedro Almodovar,Pedro Al'modovar
1,2508,film,Naked Peppers,Search Party,2014.0,"foreign, adventure, comedy",USA,NaN,16.0,NaN,The main character has learned not to invite h...,"Naked, Peppers, 2014, USA, friends, weddings, ...","Adam Palley, Brian Huskey, JB Smoove, Jason Ma...","Adam Palli, Brajan Haski, Dzh.B. Smuv, Dzhejso...",Scott Armstrong,Skot Armstrong
2,10716,film,Tactical force,Tactical Force,2011.0,"crime, foreign, thrillers, action, comedy",Canada,NaN,16.0,NaN,"Professional wrestler Steve Austin (""All or No...","Tactical, Force, 2011, Canada, bandits, gangst...","Adrian Holmes, Darren Shalavi, Jerry Wasserman...","Adrian Holms, Darren Shalavi, Dzherri Vasserma...",Adam P. Caltraro,Adam P. Kaltraro
3,7868,film,45 years old,45 Years,2015.0,"drama, foreign, melodrama",UK,NaN,16.0,NaN,"Charlotte Rampling, Tom Courtney, Geraldine Ja...","45, years, 2015, United Kingdom, marriage, lif...","Alexandra Riddleston-Barrett, Geraldine James,...","Aleksandra Riddlston-Barrett, Dzheral'din Dzhe...",By Andrew Hay,Endrju Hej
4,16268,film,Everything Solves in a Moment,NaN,1978.0,"drama, sport, soviet, melodrama",USSR,NaN,12.0,Lenfilm,The circle of her mentors and the most loyal f...,"Everything that decides, moment, 1978, USSR, s...","Aleksandr Abdulov, Aleksandr Demyanenko, Alexe...","Aleksandr Abdulov, Aleksandr Dem'janenko, Alek...",Victor Sadovsky,Viktor Sadovskij


In [7]:
df_interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1594786 entries, 0 to 1594785
Data columns (total 5 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   user_id        1594786 non-null  int64  
 1   item_id        1594786 non-null  int64  
 2   last_watch_dt  1594786 non-null  object 
 3   total_dur      1594786 non-null  int64  
 4   watched_pct    1594519 non-null  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 60.8+ MB


In [8]:
def check_date_with_formats(df, column_name, date_formats=None):
    """
    Проверка дат с учетом различных форматов
    """
    if date_formats is None:
        date_formats = ['%Y-%m-%d', '%d.%m.%Y', '%m/%d/%Y', '%Y-%m-%d %H:%M:%S']
    
    results = {}
    
    for date_format in date_formats:
        try:
            # Пробуем преобразовать с конкретным форматом
            converted = pd.to_datetime(df[column_name], format=date_format, errors='coerce')
            valid_count = converted.notna().sum()
            results[date_format] = valid_count
        except Exception:
            results[date_format] = 0
    
    # Находим лучший формат
    best_format = max(results.items(), key=lambda x: x[1])
    
    return {
        'best_format': best_format[0] if best_format[1] > 0 else None,
        'valid_count_with_best_format': best_format[1],
        'all_results': results
    }

In [9]:
def convert_to_datetime(series):
    result = pd.Series(index=series.index, dtype='object')
    idx1 = ['-' in i if i is not None else True  for i in series] # Ищет формат даты через "-" и None
    idx2 = [not i for i in idx1] # Делает замену True на False и False на True
    result[idx1] = pd.to_datetime(series[idx1], format='%Y-%m-%d')
    result[idx2] = pd.to_datetime(series[idx2], format='%d %m %Y')
    
    return result

In [10]:
df_inter_for_ds = df_interactions.drop(['total_dur'], axis=1, inplace=False).copy()

format_check = check_date_with_formats(df_inter_for_ds, 'last_watch_dt')
print("Результаты проверки форматов:")
for key, value in format_check.items():
    print(f"{key}: {value}")
     

print(df_inter_for_ds["item_id"].count())


df_inter_for_ds.loc[:, "date_w"] = convert_to_datetime (df_inter_for_ds["last_watch_dt"])


Результаты проверки форматов:
best_format: %Y-%m-%d
valid_count_with_best_format: 1594786
all_results: {'%Y-%m-%d': 1594786, '%d.%m.%Y': 0, '%m/%d/%Y': 0, '%Y-%m-%d %H:%M:%S': 0}
1594786


In [11]:
# делим на трейн и тест(валидация)
df_inter_for_ds.sort_values(by = "date_w",  ascending=False)
df_inter_for_ds.drop(columns=["last_watch_dt"],inplace=True)

days = 21
last_date = df_inter_for_ds["date_w"].max()

df_interactions_train = df_inter_for_ds[df_inter_for_ds["date_w"] < last_date - timedelta(days=days)].copy()

df_interactions_test = df_inter_for_ds[df_inter_for_ds["date_w"] >= last_date - timedelta(days=days)].copy()

In [12]:
df_inter_for_ds.columns

Index(['user_id', 'item_id', 'watched_pct', 'date_w'], dtype='object')

In [13]:
df_users[['user_id',"age","income"]]

,user_id,age,income
0,973171,age_25_34,income_60_90
1,962099,age_18_24,income_20_40
2,1047345,age_45_54,income_40_60
3,721985,age_45_54,income_20_40
4,704055,age_35_44,income_60_90
...,...,...,...
840192,339025,age_65_inf,income_0_20
840193,983617,age_18_24,income_20_40
840194,251008,NaN,NaN
840195,590706,NaN,NaN


In [14]:
from sklearn.preprocessing import LabelEncoder

# Кодируем категориальные признаки (но можно их закодировать прямо при построении датасета)
le_age = LabelEncoder()
le_inc = LabelEncoder()
le_sex = LabelEncoder()
df_users["age_en"] = le_age.fit_transform(df_users["age"])
df_users["income_en"] = le_inc.fit_transform(df_users["income"])
df_users["sex_en"] = le_inc.fit_transform(df_users["sex"])
df_users.head()

,user_id,age,income,sex,kids_flg,age_en,income_en,sex_en
0,973171,age_25_34,income_60_90,M,1,1,4,1
1,962099,age_18_24,income_20_40,M,0,0,2,1
2,1047345,age_45_54,income_40_60,F,0,3,3,0
3,721985,age_45_54,income_20_40,F,0,3,2,0
4,704055,age_35_44,income_60_90,F,0,2,4,0


In [15]:
df_interactions_train.head()

,user_id,item_id,watched_pct,date_w
0,176549,9506,72.0,2021-05-11 00:00:00
1,699317,1659,100.0,2021-05-29 00:00:00
2,656683,7107,0.0,2021-05-09 00:00:00
3,864613,7638,100.0,2021-07-05 00:00:00
4,964868,9506,100.0,2021-04-30 00:00:00


In [16]:
df_interactions_test.head()

,user_id,item_id,watched_pct,date_w
6,1016458,354,25.0,2021-08-14 00:00:00
7,884009,693,14.0,2021-08-04 00:00:00
9,203219,13582,100.0,2021-08-22 00:00:00
13,602509,496,1.0,2021-08-06 00:00:00
19,215229,7793,14.0,2021-08-01 00:00:00


In [17]:
df_interactions_test.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'date_w': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)


df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'date_w': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)

In [18]:
def check_nan (df):
    # Проверка, есть ли вообще NaN значения в DataFrame
    has_any_nans = df.isna().any().any()
    print(f"Есть ли NaN в DataFrame: {has_any_nans}")

    # Столбцы, содержащие хотя бы один NaN
    columns_with_nans = df.columns[df.isna().any()].tolist()
    print(f"Столбцы с NaN: {columns_with_nans}")

    # Количество не-NaN значений
    non_nan_count = df.count()  # По столбцам
    print("Не-NaN значений по столбцам:")
    print(non_nan_count)
    

In [19]:
check_nan (df_interactions_train)

# Удаляем все строки, где есть хотя бы один NaN
df_inter_for_ds_cleaned = df_interactions_train.dropna()

check_nan (df_inter_for_ds_cleaned)

Есть ли NaN в DataFrame: True
Столбцы с NaN: ['weight']
Не-NaN значений по столбцам:
user_id     1224560
item_id     1224560
weight      1224295
datetime    1224560
dtype: int64
Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id     1224295
item_id     1224295
weight      1224295
datetime    1224295
dtype: int64


In [20]:
check_nan (df_interactions_test)

# Удаляем все строки, где есть хотя бы один NaN
df_inter_for_test_cleaned = df_interactions_test.dropna()

check_nan (df_inter_for_test_cleaned)

Есть ли NaN в DataFrame: True
Столбцы с NaN: ['weight']
Не-NaN значений по столбцам:
user_id     370226
item_id     370226
weight      370224
datetime    370226
dtype: int64
Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id     370224
item_id     370224
weight      370224
datetime    370224
dtype: int64


In [21]:
#user_features_df= df_users[["user_id","income","age","sex"]]

 

dc = {Columns.User:[],
      "feature":[],
      'value':[],
     
      }

for index, row in df_users.iterrows():  
    dc[Columns.User].append(row["user_id"])
    dc["feature"].append("income")
    dc["value"].append(row["income_en"])

    dc["user_id"].append(row["user_id"])
    dc["feature"].append("age")
    dc["value"].append(row["age_en"])


    dc["user_id"].append(row["user_id"])
    dc["feature"].append("sex")
    dc["value"].append(row["sex_en"])
    


user_features_df = pd.DataFrame(dc)
user_features_df.head()

,user_id,feature,value
0,973171,income,4
1,973171,age,1
2,973171,sex,1
3,962099,income,2
4,962099,age,0


In [22]:
from rectools import  dataset 
# Prepare a dataset to build a model
rating = dataset.Dataset.construct(interactions_df = df_inter_for_ds_cleaned, 
                                   user_features_df=user_features_df, 
                                   #item_features_df=df_items
                                   make_dense_user_features=False
                                   )

In [23]:
from rectools.models import LightFMWrapperModel
from lightfm import LightFM

model = LightFMWrapperModel(
        # внутри модели указываем параметр no_components
        # это размезность эмбеддингов, которые выучит модель
        model=LightFM(no_components = 10)
        )


In [24]:
rating_val = dataset.Dataset.construct(interactions_df = df_inter_for_test_cleaned, 
                                   user_features_df=user_features_df, 
                                   #item_features_df=df_items
                                   make_dense_user_features=False
                                   )

In [25]:
users_unique = df_inter_for_test_cleaned[Columns.User].unique()
model.fit(rating)
recos = model.recommend(
    users=users_unique,
    dataset=rating,
    k= 20,
    filter_viewed= True
   
)

In [26]:
recos

,user_id,item_id,score,rank
0,1016458,15297,24.714350,1
1,1016458,3734,24.222328,2
2,1016458,4151,24.055174,3
3,1016458,9728,24.050138,4
4,1016458,142,23.927567,5
...,...,...,...,...
3756155,830166,16166,3.341602,16
3756156,830166,13018,3.318387,17
3756157,830166,14703,3.291361,18
3756158,830166,14461,3.249570,19


In [ ]:
# Исходные данные (уже сгруппированы)
reco_by_user = recos.groupby(Columns.User)[Columns.Item].agg(list)
actual_by_user = df_inter_for_test_cleaned.groupby(Columns.User)[Columns.Item].agg(list)

# Общая функция для вычисления метрики
def metric_ratio(r, a, metric='precision'):
    if not r:
        return 0.0
    common = len(set(r) & set(a))
    if metric == 'precision':
        return common / len(r)
    elif metric == 'recall':
        return common / len(a) if len(a) > 0 else 0.0
    elif metric == 'f1':
        p = common / len(r)
        r_val = common / len(a) if len(a) > 0 else 0.0
        return 2 * (p * r_val) / (p + r_val) if (p + r_val) > 0 else 0.0
    else:
        raise ValueError("Unsupported metric")

# Собираем DataFrame и вычисляем нужную метрику
df_metrics = pd.DataFrame({
    'reco': reco_by_user,
    'actual': actual_by_user
}).dropna()

precision_values = df_metrics.apply(lambda row: metric_ratio(row['reco'], row['actual'], 'precision'), axis=1)
precision = precision_values.mean()
print(f"Average Precision: {precision}")

recall_values = df_metrics.apply(lambda row: metric_ratio(row['reco'], row['actual'], 'recall'), axis=1)
recall = recall_values.mean()
print(f"Average Recall: {recall}")

f1_values = df_metrics.apply(lambda row: metric_ratio(row['reco'], row['actual'], 'f1'), axis=1)
f1 = f1_values.mean()
print(f"Average F1: {f1}")

Average Precision: 0.01651473845629579
Average Recall: 0.2183813044790131
Average F1: 0.029688113309083903


Эта модель показала себя лучше остальных, имея показатель псевдо-recall в 21%.

In [ ]:


# d = {"user_id": [], "recos": [], 'actual': []} 
# recall_sum = 0
# user_count = 0

# for u in users_unique:
#     d["user_id"].append(u)
#     r = (recos[Columns.Item][recos[Columns.User] == u]).to_list()
#     d["recos"].append(r)
#     a = (df_inter_for_test_cleaned[Columns.Item][df_inter_for_test_cleaned[Columns.User] == u]).to_list()
#     d["actual"].append(a)

#     print(f"user {u}:\n rec {r} \n act {a}")

#     # Для recall_score нужно преобразовать данные в бинарный формат
#     if len(a) > 0 and len(r) > 0:
#         # Создаем бинарные векторы для расчета recall
#         all_items = list(set(r + a))
#         y_true = [1 if item in a else 0 for item in all_items]
#         y_pred = [1 if item in r else 0 for item in all_items]
        
#         user_recall = recall_score(y_true, y_pred, zero_division=0)
#         recall_sum += user_recall
#         user_count += 1
#     else:
#         # Если нет данных для пользователя, пропускаем или добавляем 0
#         user_count += 1

# recall = recall_sum / user_count if user_count > 0 else 0

# print(f"Average Recall: {recall}")